# Load genomic data (relevant columns only), do one hot encoding, save for training/validation

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import random

import json
import yaml
import shutil

import random
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
!pwd

/project/ich248_uksr/DMYTRO/ZCOR/RESEARCHINGS/COLORADO_BIOBANK/ILD/zed_colorado/ZEBRA_GENDRIVERS_CLASSIFIER/v2


In [3]:
GENDIR = "/project/ich248_uksr/IXC/LSM_genome"

In [4]:
with open(f"{GENDIR}/genomicdata.csv", 'r') as f:
    RAW_COLS = f.readline().replace('\n','').split(',')

In [25]:
DRIVERS = pd.read_csv(f"{GENDIR}/biological_drivers.csv")
DRIVERS_2 = pd.read_csv(f"MORE_LOCI.csv")
DRIVERS_3 = pd.read_csv("TOBACCO_SNPS.csv")
print(DRIVERS_2.shape)
print(DRIVERS_3.shape)
# Both direct SNP names and chr:basepair formats to look for column_names
SNPS = set(DRIVERS_2.SNP) | \
       set(DRIVERS_3.genomic_header_name) | \
set([f"{r.CHR}:{r.BP}" for _, r in DRIVERS_2.iterrows()])
NEW_DRIVER_COLUMNS = [i for i in RAW_COLS if i[:-2] in SNPS or i.split("-")[0] in SNPS] # All columns end with _{somenucleotide}
print(len(NEW_DRIVER_COLUMNS))

(3113, 7)
(692, 13)
560


In [30]:
DRIVERS.sample(2)

,rsid,gene_or_region,column_name,line,notes
14,rs4077759,MUC2/TOLLIP region,rs4077759_T,1019588,11p15 signal; may not be independent of MUC5B
5,rs1278769,ATP11A,rs1278769_G,1235606,GWAS IPF locus


In [28]:
DRIVERS_2.sample(2)

,source_sheet,SNP,CHR,BP,P,Base,Gene
714,gwas_wgs_prs_7E-4_SNPs,rs2301594,6,16754366,0.000254,1,ATXN1
2725,"gwas p<7E-4, keep reseq SNPs",rs76287794,13,77515486,0.000494,1,-


In [31]:
DRIVERS_3.sample(2)

,source_sheet,SNP,CHR,BP,P,Base,Gene,genomic_header_name,match_type,genomic_header_CHR,genomic_header_BP,distance_bp,nearby_threshold_bp
394,gwas_wgs_prs_7E-4_SNPs,rs112996625,4.0,90041999.0,0.000336,1.0,RP11-84C13.2,JHU_4.90042326_A,NEAREST_WITHIN_1KB,4.0,90042326.0,327.0,1000.0
60,gwas_wgs_prs_7E-4_SNPs,rs918174,18.0,3736629.0,0.000005,1.0,DLGAP1,rs918174_T,EXACT_RSID,18.0,3736629.0,0.0,NaN


### Only load the columns you need - would be much faster with .parquet file

#### Loading loci from both csvs I received

#### this cell takes some time and ~65GB of RAM

In [32]:
try:
    GEN_DATA = pl.read_csv(
        f"{GENDIR}/genomicdata.csv", columns = ["FID"] + list(set(NEW_DRIVER_COLUMNS + list(DRIVERS['column_name'])))
    )
except ColumnNotFoundError:
    print("ColumnNotFoundError")

In [33]:
GD = GEN_DATA.to_pandas().rename(columns = {'FID': 'patient_id'})

In [34]:
GD.shape

(19651, 583)

In [35]:
GD[list(GD.columns)[9]].value_counts()

JHU_1.22343297_C
2.0    16942
1.0     2588
0.0      106
Name: count, dtype: int64

In [36]:
GD.iloc[:4,:4]

,patient_id,JHU_1.3296950_C,JHU_1.3324416_C,JHU_1.8017528_C
0,1230142395,2.0,2.0,2.0
1,6674359887,0.0,2.0,2.0
2,6802160313,2.0,2.0,2.0
3,6489473597,2.0,2.0,2.0


In [37]:
# One-hot encode all genotype columns, forcing categories 0, 1, 2
ID = GD[["patient_id"]]

X = GD.drop(columns="patient_id").apply(
    lambda x: pd.Categorical(x, categories=[0, 1, 2])
)

X = pd.get_dummies(X, dtype=int)

GD_OHE = pd.concat([ID, X], axis=1)

In [40]:
GD_OHE.columns = [i.replace(":","-") for i in GD_OHE.columns]

In [41]:
print(GD_OHE.shape)
display(GD_OHE.head(3))
GD_OHE.to_csv("ILD_TOP_DRIVERS_DATA.csv", index = False)

(19651, 1747)


,patient_id,JHU_1.3296950_C_0,JHU_1.3296950_C_1,JHU_1.3296950_C_2,JHU_1.3324416_C_0,JHU_1.3324416_C_1,JHU_1.3324416_C_2,JHU_1.8017528_C_0,JHU_1.8017528_C_1,JHU_1.8017528_C_2,...,JHU_22.49419625_A_2,SmokingTobaccoUse_0,SmokingTobaccoUse_1,SmokingTobaccoUse_2,SmokelessTobaccoUse_0,SmokelessTobaccoUse_1,SmokelessTobaccoUse_2,TobaccoUse_0,TobaccoUse_1,TobaccoUse_2
0,1230142395,0,0,1,0,0,1,0,0,1,...,1,0,0,0,0,0,0,0,0,0
1,6674359887,1,0,0,0,0,1,0,0,1,...,1,0,0,0,0,0,0,0,0,0
2,6802160313,0,0,1,0,0,1,0,0,1,...,1,0,0,0,0,0,0,0,0,0
